In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import inception_v3, Inception_V3_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"

what is Inception-v3?

In [9]:
# Load pretrained Inception-v3
weights = Inception_V3_Weights.DEFAULT
inception = inception_v3(weights=weights, aux_logits=True).to(device)
inception.eval()

# Replace final classifier with identity so output is 2048-dim features
inception.fc = nn.Identity()

# Torchvision Inception expects 299x299 images and ImageNet normalization
preprocess = weights.transforms()

In [12]:
@torch.no_grad()
def extract_inception_features(images, batch_size=256):
    """
    images: torch.Tensor of shape [N, 3, H, W]

    Expected input range:
      - [0, 1] float images, or
      - [-1, 1] float images if you uncomment the conversion below

    returns:
      features of shape [N, 2048]
    """
    inception.eval()
    features = []

    for i in range(0, len(images), batch_size):
        x = images[i:i + batch_size].to(device)

        # If your generated images are in [-1, 1], uncomment this:
        # x = (x.clamp(-1, 1) + 1) / 2

        # Resize + ImageNet normalize
        x = preprocess(x)

        feat = inception(x)

        # In case torchvision returns an InceptionOutputs object
        if not isinstance(feat, torch.Tensor):
            feat = feat.logits

        features.append(feat.cpu())

    return torch.cat(features, dim=0)


# Example:
# real_images: [N, 3, 32, 32], float in [0, 1]
# fake_images: [N, 3, 32, 32], float in [0, 1]

N = 100

real_images = torch.rand(N, 3, 32, 32)
fake_images = torch.rand(N, 3, 32, 32)

real_features = extract_inception_features(real_images)
fake_features = extract_inception_features(fake_images)

print(real_features.shape)  # [N, 2048]
print(fake_features.shape)  # [N, 2048]

torch.Size([100, 2048])
torch.Size([100, 2048])


In [14]:
def ploynomial_kernel(x, y, degree=3):
    scale = 1.0 / x.shape[1]
    return (scale * x @ y.T + 1) ** degree

## KID:
- take N real and N fake images
- extract a k-dimensional (2048, say) feature vector for each image (using torchvision) $\quad\implies\phi$
- get pairwise kernel similarities $\quad\implies k(x_i, y_i)$
    - 3 sets: (real, real), (fake, fake) and (real, fake)
    - feed those 3 similarities through formula

In [27]:
def kid_estimate(real_features, fake_features):
    k_xx = ploynomial_kernel(real_features, real_features)
    k_yy = ploynomial_kernel(fake_features, fake_features)
    k_xy = ploynomial_kernel(real_features, fake_features)

    sum_k_xx = k_xx.sum() - k_xx.diag().sum()
    sum_k_yy = k_yy.sum() - k_yy.diag().sum()
    sum_k_xy = k_xy.sum()

    mmd = (sum_k_xx / (N * (N - 1))) + (sum_k_yy / (N * (N - 1))) - (2 * sum_k_xy / (N * N))
    return mmd.item()

kid_score = kid_estimate(real_features, fake_features)
print("KID Score:", kid_score)

KID Score: -0.0003790855407714844


In [36]:
def kid_score(real_features, fake_features, subsets=5, subset_size=2):
    device = real_features.device
    scores = []
    
    n_real = real_features.shape[0]
    n_fake = fake_features.shape[0]
    
    if n_real < subset_size or n_fake < subset_size:
        raise ValueError("subset_size must be no larger than both feature sets.")
    
    for _ in range(subsets):
        real_idx = torch.randperm(subset_size, device=device)
        fake_idx = torch.randperm(subset_size, device=device)
        
        x = real_features[real_idx]
        y = fake_features[fake_idx]
        
        kid_est = kid_estimate(x, y)
        # print(f"KID estimate for subset: {kid_est}")
        scores.append(kid_est)
        # print(scores)
    
    # print(type(scores))
    scores = torch.tensor(scores)
    # print(scores.shape)
    return scores.mean().item(), scores.std().item()

In [37]:
kid_score(real_features, fake_features)

(-0.0006260976078920066, 1.0493535207434235e-10)

## Torch-Fidelity Validation

Use this section as an external reference implementation for FID/KID/ISC. It evaluates image folders, so it matches the output format from `scripts/generate_eval_samples.py`.

Important convention: `input1` is the generated/evaluated image folder and `input2` is the real/reference image folder.

In [ ]:
# If torch-fidelity is not installed in this kernel, uncomment and run this once.
# %pip install torch-fidelity

from pathlib import Path
import json

import torch
from torch_fidelity import calculate_metrics
from torchvision import datasets, transforms
from torchvision.utils import save_image
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
DEVICE_HAS_CUDA = torch.cuda.is_available()
print(f"CUDA available: {DEVICE_HAS_CUDA}")

In [ ]:
# Real/reference image folder. This uses CIFAR-10 test images saved as plain PNGs.
REAL_DIR = PROJECT_ROOT / "outputs" / "eval" / "reference" / "cifar10_test_32"

# Generated image folder. Change this to the model/seed you want to validate.
FAKE_DIR = PROJECT_ROOT / "outputs" / "eval" / "samples" / "ddpm_cifar10_seed0"

METRICS_OUT = PROJECT_ROOT / "outputs" / "eval" / "metrics" / f"{FAKE_DIR.name}_torch_fidelity.json"

print("REAL_DIR:", REAL_DIR)
print("FAKE_DIR:", FAKE_DIR)
print("METRICS_OUT:", METRICS_OUT)

In [ ]:
def count_pngs(path: Path) -> int:
    return len(list(path.glob("*.png")))


def export_cifar10_reference(output_dir: Path, *, split: str = "test", image_size: int = 32, force: bool = False) -> None:
    existing = count_pngs(output_dir)
    if existing > 0 and not force:
        print(f"Reference folder already has {existing} PNGs: {output_dir}")
        return

    output_dir.mkdir(parents=True, exist_ok=True)
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),  # saves as [0, 1] image-space PNG, not normalized [-1, 1]
    ])
    dataset = datasets.CIFAR10(
        root=PROJECT_ROOT / "data",
        train=(split == "train"),
        download=True,
        transform=transform,
    )

    for index, (image, _) in enumerate(tqdm(dataset, desc=f"export cifar10-{split}")):
        save_image(image, output_dir / f"{index:06d}.png")

    print(f"Wrote {count_pngs(output_dir)} reference PNGs to {output_dir}")


export_cifar10_reference(REAL_DIR, split="test", image_size=32)

print("real PNG count:", count_pngs(REAL_DIR))
print("fake PNG count:", count_pngs(FAKE_DIR))

In [ ]:
if count_pngs(FAKE_DIR) == 0:
    raise FileNotFoundError(
        f"No generated PNGs found in {FAKE_DIR}. "
        "Run something like: python scripts/generate_eval_samples.py ddpm 0"
    )

METRICS_OUT.parent.mkdir(parents=True, exist_ok=True)

torch_fidelity_metrics = calculate_metrics(
    input1=str(FAKE_DIR),  # generated/evaluated images
    input2=str(REAL_DIR),  # real/reference images
    cuda=DEVICE_HAS_CUDA,
    isc=True,
    fid=True,
    kid=True,
    kid_subsets=100,
    kid_subset_size=1000,
    verbose=True,
)

print(json.dumps(torch_fidelity_metrics, indent=2))

with METRICS_OUT.open("w", encoding="utf-8") as handle:
    json.dump(torch_fidelity_metrics, handle, indent=2)

print(f"Saved torch-fidelity metrics to {METRICS_OUT}")

### Comparing Against The Custom KID Code

The custom KID result should only match torch-fidelity closely if the protocol is aligned:

- same real and generated image sets
- same feature extractor and feature layer
- same resize and normalization before Inception
- same polynomial kernel settings: degree 3, gamma `1 / feature_dim`, coef 1
- same KID subset count and subset size
- same unbiased MMD estimator

For validation, start with a small generated folder and compare the KID order of magnitude. Exact equality is not expected until the feature extractor and subset sampling protocol are identical.